# Ejercicios de XML con ElementTree

**Acceso a Datos · Tema 1: Manejo de ficheros**

> ⚠️ **Contenido generado con ayuda de IA.** Los enunciados y las soluciones se han generado con ayuda de una
> herramienta de inteligencia artificial y se han ejecutado antes de publicarlos, pero pueden contener errores.
> Contrasta lo que leas con la [documentación oficial de ElementTree](https://docs.python.org/3/library/xml.etree.elementtree.html).

Diez ejercicios en orden creciente de dificultad. Solo se usa el módulo `xml.etree.ElementTree` y otros módulos de la
biblioteca estándar (`pathlib`, `csv`, `json`, `zipfile`, `dataclasses`, `urllib`…). No hace falta instalar nada.

| Nº | Ejercicio | Dificultad | Lo que se practica |
|---|---|:---:|---|
| 1 | Inventario de un taller de bicicletas | ★ | `parse`, `getroot`, `find`, `findtext`, atributos |
| 2 | Consultas a un horario de trenes | ★ | XPath de ElementTree y sus límites |
| 3 | Clasificación de una liga | ★ | Crear documentos: `Element`, `SubElement`, `indent`, `write` |
| 4 | Limpieza de una lista de reproducción | ★★ | Modificar el árbol: `remove`, `set`, `.tag`, ordenar |
| 5 | Lecturas de sensores con datos sucios | ★★ | Conversión robusta y `ParseError` |
| 6 | Ruta de senderismo en GPX | ★★ | Espacios de nombres |
| 7 | Fusión de pedidos de dos sucursales | ★★ | Transformar una jerarquía en otra |
| 8 | Mirar dentro de un `.docx` | ★★★ | XML real dentro de un ZIP, varios espacios de nombres |
| 9 | Un fichero XML enorme | ★★★ | `iterparse` frente a `parse`: memoria y tiempo |
| 10 | Proyecto: el tiempo en Castilla y León | ★★★ | Datos reales de AEMET: descarga, caché, codificación, modelo |

**Estructura de carpetas.** El cuaderno espera esta organización (los ficheros de `salida/` los crean los ejercicios):

```
ejercicios_xml/
├── Ejercicios_XML_ElementTree.ipynb
├── datos/
│   ├── taller.xml, horario.xml, playlist.xml
│   ├── estacion.xml, estacion_rota.xml
│   ├── ruta.gpx, pedidos_norte.xml, pedidos_sur.xml, memoria.docx
│   └── aemet/localidad_47186.xml      ← copia de la predicción real de Valladolid
└── salida/
```

Ejecuta primero la celda siguiente: importa ElementTree y define las rutas que usan todos los ejercicios.

In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path

DATOS = Path("datos")
SALIDA = Path("salida")
SALIDA.mkdir(exist_ok=True)

---
## Ejercicio 1. Inventario de un taller de bicicletas ★

El fichero `datos/taller.xml` guarda el inventario de un taller. Cada bicicleta es un elemento `<bici>` con los
atributos `id` y `tipo`, y los elementos `marca`, `modelo`, `precio` y `stock`:

```xml
<bici id="B001" tipo="montaña">
  <marca>Orbea</marca>
  <modelo>Alma M30</modelo>
  <precio moneda="EUR">1899.00</precio>
  <stock>3</stock>
</bici>
```

Escribe un programa que:

1. Muestre el nombre y la ciudad del taller, que son atributos del elemento raíz.
2. Muestre una tabla alineada con el id, tipo, marca, modelo, precio (con dos decimales, alineado a la derecha) y stock
   de cada bici.
3. Si una bici no tiene atributo `tipo`, muestre «sin clasificar».
4. Si una bici no tiene elemento `stock`, lo cuente como 0 **y avise al final** de qué bicis no tienen el dato.
5. Calcule el valor total del inventario (precio × stock) y diga cuál es la bici más cara.

**Pistas.** `elemento.get("atributo", valor_por_defecto)` y `elemento.findtext("hijo", valor_por_defecto)`.
Piensa si te conviene usar el valor por defecto de `findtext` en el apartado 4.

### Solución

In [2]:
arbol = ET.parse(DATOS / "taller.xml")
taller = arbol.getroot()

print(f"Taller: {taller.get('nombre')} ({taller.get('ciudad')})\n")
print(f"{'ID':<5} {'Tipo':<15} {'Marca':<10} {'Modelo':<16} {'Precio':>9} {'Stock':>5}")
print("-" * 65)

valor_total = 0.0
sin_dato_stock = []

for bici in taller.findall("bici"):
    tipo = bici.get("tipo", "sin clasificar")
    precio = float(bici.findtext("precio"))

    texto_stock = bici.findtext("stock")      # None si el elemento no existe
    if texto_stock is None:
        sin_dato_stock.append(bici.get("id"))
        stock = 0
    else:
        stock = int(texto_stock)

    valor_total += precio * stock
    print(f"{bici.get('id'):<5} {tipo:<15} {bici.findtext('marca'):<10} "
          f"{bici.findtext('modelo'):<16} {precio:>9.2f} {stock:>5}")

mas_cara = max(taller.findall("bici"), key=lambda b: float(b.findtext("precio")))

print("-" * 65)
print(f"Valor total del inventario: {valor_total:,.2f} €")
print(f"Bici más cara: {mas_cara.findtext('marca')} {mas_cara.findtext('modelo')} "
      f"({float(mas_cara.findtext('precio')):.2f} €)")
if sin_dato_stock:
    print(f"Aviso: sin dato de stock en {', '.join(sin_dato_stock)}")

Taller: Radios y Piñones (Valladolid)

ID    Tipo            Marca      Modelo              Precio Stock
-----------------------------------------------------------------
B001  montaña         Orbea      Alma M30           1899.00     3
B002  carretera       BH         Ultralight 7.0     2499.90     1
B003  urbana          Brompton   C Line Explore     1650.00     4
B004  gravel          Mondraker  Dusty RR           3299.00     0
B005  montaña         Megamo     Factory 10          899.00     0
B006  eléctrica       Orbea      Vibe H30           2799.00     2
B007  sin clasificar  Conor      WRC Pro             649.50     6
-----------------------------------------------------------------
Valor total del inventario: 24,291.90 €
Bici más cara: Mondraker Dusty RR (3299.00 €)
Aviso: sin dato de stock en B004


**Comentarios**

- Con `bici.findtext("stock", "0")` el programa también funcionaría, pero no podríamos distinguir «stock 0» de
  «no hay dato», y el apartado 4 pide avisar. Por eso se pide `findtext` **sin** valor por defecto y se comprueba `None`.
- `findtext` devuelve `""` (no `None`) si el elemento existe pero está vacío, como `<stock/>`.
- El atributo `moneda` de `<precio>` se ignora porque todas las bicis están en euros. Se leería con
  `bici.find("precio").get("moneda")`.

---
## Ejercicio 2. Consultas a un horario de trenes ★

El fichero `datos/horario.xml` contiene los trenes que salen de Valladolid un día:

```xml
<tren numero="18021" tipo="MD">
  <destino>Salamanca</destino>
  <paradas>
    <parada><estacion>Valladolid-Campo Grande</estacion><hora>07:40</hora></parada>
    <parada><estacion>Medina del Campo</estacion><hora>08:12</hora></parada>
    ...
  </paradas>
  <servicios/>            <!-- puede tener <cafeteria/> y <wifi/>, o no existir -->
</tren>
```

Resuelve cada consulta **con una sola llamada a `findall` o `find`** siempre que se pueda, y con Python cuando
ElementTree no lo permita:

1. Número y hora de salida de los trenes de tipo AVE.
2. Trenes que paran en Medina del Campo.
3. Última parada de cada tren y hora de llegada.
4. Trenes con cafetería.
5. Trenes **sin** cafetería.
6. El tercer tren del horario.
7. Lista de todas las estaciones distintas, ordenada alfabéticamente.

Prueba también `horario.findall(".//tren[servicios/cafeteria]")` y explica qué ocurre.

**Resumen del XPath que admite ElementTree**

| Expresión | Selecciona |
|---|---|
| `tag`, `*`, `.`, `..`, `//` | hijos con esa etiqueta, todos los hijos, el propio nodo, el padre, cualquier descendiente |
| `[@atr]`, `[@atr='v']`, `[@atr!='v']` | elementos con ese atributo, con ese valor o con otro valor |
| `[tag]` | elementos que tienen un **hijo directo** `tag` |
| `[tag='texto']`, `[.='texto']` | hijo (o el propio elemento) cuyo texto completo es `texto` |
| `[1]`, `[last()]`, `[last()-1]` | posición entre los hermanos con la misma etiqueta (empieza en 1) |

### Solución

In [3]:
horario = ET.parse(DATOS / "horario.xml").getroot()

print("1. Trenes AVE")
for tren in horario.findall("tren[@tipo='AVE']"):
    print(f"   {tren.get('numero')} sale a las {tren.findtext('paradas/parada[1]/hora')}")

print("2. Paran en Medina del Campo")
# '..' sube al padre: de <parada> a <paradas> y de <paradas> a <tren>
for tren in horario.findall(".//parada[estacion='Medina del Campo']/../.."):
    print(f"   {tren.get('numero')} ({tren.get('tipo')}) con destino {tren.findtext('destino')}")

print("3. Última parada de cada tren")
for tren in horario.findall("tren"):
    ultima = tren.find("paradas/parada[last()]")
    print(f"   {tren.get('numero'):>6}: {ultima.findtext('estacion')} a las {ultima.findtext('hora')}")

print("4. Con cafetería")
con_cafeteria = horario.findall("tren/servicios[cafeteria]/..")
print("  ", [t.get("numero") for t in con_cafeteria])

print("5. Sin cafetería (ElementTree no tiene not(): se filtra en Python)")
sin_cafeteria = [t for t in horario.findall("tren") if t.find("servicios/cafeteria") is None]
print("  ", [t.get("numero") for t in sin_cafeteria])

print("6. Tercer tren")
tercero = horario.find("tren[3]")
print(f"   {tercero.get('numero')} con destino {tercero.findtext('destino')}")

print("7. Estaciones distintas")
estaciones = sorted({e.text for e in horario.iter("estacion")})
print("  ", ", ".join(estaciones))

1. Trenes AVE
   4071 sale a las 07:05
   4083 sale a las 09:15
2. Paran en Medina del Campo
   18021 (MD) con destino Salamanca
   18307 (Regional) con destino Ávila
   18951 (MD) con destino Medina del Campo
3. Última parada de cada tren
     4071: Madrid-Chamartín a las 08:02
    18021: Salamanca a las 09:10
     4083: Madrid-Chamartín a las 10:10
    00631: León a las 11:30
    18307: Ávila a las 13:45
     4165: Burgos-Rosa Manzano a las 16:25
    18951: Medina del Campo a las 19:42
4. Con cafetería
   ['4071', '00631', '4165']
5. Sin cafetería (ElementTree no tiene not(): se filtra en Python)
   ['18021', '4083', '18307', '18951']
6. Tercer tren
   4083 con destino Madrid-Chamartín
7. Estaciones distintas
   Arévalo, Burgos-Rosa Manzano, Cantalapiedra, León, Madrid-Chamartín, Medina del Campo, Palencia, Salamanca, Segovia-Guiomar, Valdestillas, Valladolid-Campo Grande, Ávila


In [4]:
# ¿Y la consulta "natural" para el apartado 4?
try:
    horario.findall(".//tren[servicios/cafeteria]")
except SyntaxError as e:
    print("SyntaxError:", e)

SyntaxError: invalid predicate


**Comentarios**

- En los predicados `[tag]` solo se admite un **hijo directo**. `tren[servicios/cafeteria]` no es válido; la
  alternativa es poner el predicado en el nivel correcto y subir con `..`: `tren/servicios[cafeteria]/..`.
- ElementTree no tiene funciones como `not()`, `contains()` o `count()`. Lo que no se puede expresar se filtra en
  Python, como en el apartado 5. Con `lxml` (Anexo B) sí se podría usar XPath 1.0 completo.
- `..` funciona dentro de `findall`, pero un elemento no guarda una referencia a su padre: `tren.find("..")`
  devuelve `None` porque `tren` es el punto de partida de la búsqueda.
- Ordenar con `sorted` pone «Ávila» al final, porque compara códigos Unicode. Para un orden alfabético español
  habría que usar `locale.strxfrm` como clave.

---
## Ejercicio 3. Clasificación de una liga ★

Tienes los resultados de las tres primeras jornadas de una liga provincial en una lista de Python (ejecuta la
celda siguiente). Genera el fichero `salida/clasificacion.xml` con la clasificación.

1. Calcula para cada equipo: partidos jugados, ganados, empatados y perdidos, goles a favor y en contra, diferencia de
   goles y puntos (3 por victoria, 1 por empate).
2. Ordena por puntos, después por diferencia de goles, después por goles a favor y, en último caso, por nombre.
3. Diseña la estructura del XML: decide qué datos van como **atributo** y cuáles como **elemento**, y justifícalo en
   una celda de texto.
4. El fichero debe llevar declaración XML, estar en UTF-8 y estar sangrado.
5. Vuelve a leer el fichero generado y muestra los tres primeros.

**Pistas.** `ET.Element`, `ET.SubElement`, `ET.indent(arbol)` y
`arbol.write(ruta, encoding="utf-8", xml_declaration=True)`. Los valores de atributos y textos deben ser cadenas.

In [5]:
RESULTADOS = [
    {"jornada": 1, "local": "CD Pinares",    "visitante": "Atlético Duero", "gl": 2, "gv": 1},
    {"jornada": 1, "local": "UD Meseta",     "visitante": "CF Castillo",    "gl": 0, "gv": 0},
    {"jornada": 1, "local": "Racing Páramo", "visitante": "SD Ribera",      "gl": 3, "gv": 2},
    {"jornada": 2, "local": "Atlético Duero","visitante": "UD Meseta",      "gl": 1, "gv": 1},
    {"jornada": 2, "local": "CF Castillo",   "visitante": "Racing Páramo",  "gl": 2, "gv": 0},
    {"jornada": 2, "local": "SD Ribera",     "visitante": "CD Pinares",     "gl": 0, "gv": 1},
    {"jornada": 3, "local": "CD Pinares",    "visitante": "CF Castillo",    "gl": 1, "gv": 3},
    {"jornada": 3, "local": "UD Meseta",     "visitante": "Racing Páramo",  "gl": 2, "gv": 2},
    {"jornada": 3, "local": "SD Ribera",     "visitante": "Atlético Duero", "gl": 1, "gv": 0},
]

### Solución

In [6]:
from datetime import datetime


def nueva_fila():
    return {"pj": 0, "g": 0, "e": 0, "p": 0, "gf": 0, "gc": 0}


def calcular_tabla(resultados):
    tabla = {}
    for r in resultados:
        for equipo, favor, contra in ((r["local"], r["gl"], r["gv"]),
                                      (r["visitante"], r["gv"], r["gl"])):
            fila = tabla.setdefault(equipo, nueva_fila())
            fila["pj"] += 1
            fila["gf"] += favor
            fila["gc"] += contra
            if favor > contra:
                fila["g"] += 1
            elif favor == contra:
                fila["e"] += 1
            else:
                fila["p"] += 1
    for fila in tabla.values():
        fila["dg"] = fila["gf"] - fila["gc"]
        fila["puntos"] = 3 * fila["g"] + fila["e"]
    return sorted(tabla.items(),
                  key=lambda par: (-par[1]["puntos"], -par[1]["dg"], -par[1]["gf"], par[0]))


def clasificacion_a_xml(clasificacion, jornada):
    raiz = ET.Element("clasificacion", competicion="Liga Provincial", jornada=str(jornada),
                      generado=datetime.now().isoformat(timespec="seconds"))
    for posicion, (nombre, f) in enumerate(clasificacion, start=1):
        equipo = ET.SubElement(raiz, "equipo", posicion=str(posicion), nombre=nombre)
        ET.SubElement(equipo, "puntos").text = str(f["puntos"])
        partidos = ET.SubElement(equipo, "partidos", jugados=str(f["pj"]))
        ET.SubElement(partidos, "ganados").text = str(f["g"])
        ET.SubElement(partidos, "empatados").text = str(f["e"])
        ET.SubElement(partidos, "perdidos").text = str(f["p"])
        ET.SubElement(equipo, "goles", favor=str(f["gf"]), contra=str(f["gc"]),
                      diferencia=f"{f['dg']:+d}")
    return ET.ElementTree(raiz)


clasificacion = calcular_tabla(RESULTADOS)
arbol = clasificacion_a_xml(clasificacion, jornada=max(r["jornada"] for r in RESULTADOS))
ET.indent(arbol)
arbol.write(SALIDA / "clasificacion.xml", encoding="utf-8", xml_declaration=True)

print((SALIDA / "clasificacion.xml").read_text(encoding="utf-8")[:700], "...")

<?xml version='1.0' encoding='utf-8'?>
<clasificacion competicion="Liga Provincial" jornada="3" generado="2026-10-19T15:16:42">
  <equipo posicion="1" nombre="CF Castillo">
    <puntos>7</puntos>
    <partidos jugados="3">
      <ganados>2</ganados>
      <empatados>1</empatados>
      <perdidos>0</perdidos>
    </partidos>
    <goles favor="5" contra="1" diferencia="+4" />
  </equipo>
  <equipo posicion="2" nombre="CD Pinares">
    <puntos>6</puntos>
    <partidos jugados="3">
      <ganados>2</ganados>
      <empatados>0</empatados>
      <perdidos>1</perdidos>
    </partidos>
    <goles favor="4" contra="4" diferencia="+0" />
  </equipo>
  <equipo posicion="3" nombre="Racing Páramo">
     ...


In [7]:
# Comprobación: volver a leer el fichero generado
leida = ET.parse(SALIDA / "clasificacion.xml").getroot()
for equipo in leida.findall("equipo[@posicion]")[:3]:
    goles = equipo.find("goles")
    print(f"{equipo.get('posicion')}. {equipo.get('nombre'):<15} {equipo.findtext('puntos'):>2} pts  "
          f"(dif. {goles.get('diferencia')})")

1. CF Castillo      7 pts  (dif. +4)
2. CD Pinares       6 pts  (dif. +0)
3. Racing Páramo    4 pts  (dif. -1)


In [8]:
# Error típico: valores que no son cadenas
prueba = ET.Element("equipo", posicion=1)          # ← entero, no cadena
try:
    ET.tostring(prueba)
except TypeError as e:
    print("TypeError:", e)

TypeError: cannot serialize 1 (type int)


**Justificación del diseño**

- **Atributos** para lo que identifica o describe al elemento: `posicion` y `nombre` del equipo, `competicion` y
  `jornada` de la clasificación. Son valores simples que no se van a desglosar.
- **Elementos** para los datos que se consultan o podrían crecer: `puntos` y `partidos`, que agrupa ganados,
  empatados y perdidos. Si mañana se añaden, por ejemplo, partidos en casa y fuera, cabe en esa estructura.
- `goles` se modela como un elemento con tres atributos porque los tres valores forman una unidad y nunca se
  desglosan más. Es una decisión discutible: otra solución igual de válida usa tres elementos hijos.
- ElementTree no comprueba tipos al crear el árbol: el error de un entero en un atributo aparece al **serializar**.

---
## Ejercicio 4. Limpieza de una lista de reproducción ★★

El fichero `datos/playlist.xml` tiene una lista de canciones con título, artista, duración (`m:ss`) y valoración
de 1 a 5. Escribe un programa que haga estos cambios **en este orden** y guarde el resultado en
`salida/playlist_limpia.xml` (el original no se modifica):

1. Elimina las canciones de menos de 60 segundos.
2. Añade el atributo `favorita="si"` a las canciones con valoración 4 o 5.
3. Cambia la etiqueta `<cancion>` por `<pista>`.
4. Ordena las pistas por duración, de menor a mayor.
5. Añade a la raíz los atributos `total_pistas` y `duracion_total` (en formato `m:ss`).

Antes de resolverlo, ejecuta la celda de «versión con error» y explica por qué no borra todas las canciones cortas.

In [9]:
# Versión con error: borrar mientras se recorre la lista de hijos
def segundos(texto):
    minutos, segs = texto.split(":")
    return int(minutos) * 60 + int(segs)


raiz = ET.parse(DATOS / "playlist.xml").getroot()
for cancion in raiz:
    if segundos(cancion.findtext("duracion")) < 60:
        raiz.remove(cancion)

for cancion in raiz:
    print(cancion.findtext("duracion"), cancion.findtext("titulo"))

4:12 Compilando a medianoche
3:05 Commit sin mensaje
0:15 Tic
2:30 Hola, mundo
5:47 Merge conflict
3:59 Despliegue en viernes


### Solución

**Por qué falla.** Recorrer `raiz` es recorrer su lista de hijos por posición. Al borrar «Interludio» (posición 3),
«Tic» pasa a ocupar la posición 3, pero el bucle ya va a la 4: «Tic» nunca se comprueba. La solución es recorrer una
**copia** de la lista con `list(raiz)` o `raiz.findall(...)`.

In [10]:
def segundos(texto):
    minutos, segs = texto.split(":")
    return int(minutos) * 60 + int(segs)


def a_mmss(total):
    return f"{total // 60}:{total % 60:02d}"


arbol = ET.parse(DATOS / "playlist.xml")
raiz = arbol.getroot()

# 1. Borrar las cortas recorriendo una copia
eliminadas = []
for cancion in list(raiz):
    if segundos(cancion.findtext("duracion")) < 60:
        eliminadas.append(cancion.findtext("titulo"))
        raiz.remove(cancion)

# 2. Marcar favoritas
for cancion in raiz.findall("cancion"):
    if int(cancion.findtext("valoracion")) >= 4:
        cancion.set("favorita", "si")

# 3. Renombrar
for cancion in raiz.findall("cancion"):
    cancion.tag = "pista"

# 4. Ordenar: se sustituyen todos los hijos por la lista ordenada
raiz[:] = sorted(raiz, key=lambda p: segundos(p.findtext("duracion")))

# 5. Totales en la raíz
raiz.set("total_pistas", str(len(raiz)))
raiz.set("duracion_total", a_mmss(sum(segundos(p.findtext("duracion")) for p in raiz)))

ET.indent(arbol)                       # rehace la sangría tras borrar y reordenar
arbol.write(SALIDA / "playlist_limpia.xml", encoding="utf-8", xml_declaration=True)

print("Eliminadas:", ", ".join(eliminadas))
print(f"{raiz.get('total_pistas')} pistas, {raiz.get('duracion_total')} en total\n")
for p in raiz:
    estrella = "★" if p.get("favorita") == "si" else " "
    print(f"{estrella} {p.findtext('duracion'):>5}  {p.findtext('titulo')} ({p.findtext('artista')})")

Eliminadas: Intro (versión radio), Interludio, Tic
5 pistas, 19:33 en total

   2:30  Hola, mundo (La Recursiva)
   3:05  Commit sin mensaje (Los Bucles)
★  3:59  Despliegue en viernes (Stack Overflow Band)
★  4:12  Compilando a medianoche (Null Pointer)
★  5:47  Merge conflict (Null Pointer)


**Comentarios**

- El orden importa: «Tic» tiene valoración 5, pero no llega a ser favorita porque se elimina antes.
- `raiz[:] = ...` reemplaza los hijos usando la sintaxis de listas. Un `Element` se comporta como una lista de sus
  hijos: admite `len`, índices, `append`, `insert`, `remove` y asignación por porciones.
- Después de borrar o mover elementos, los espacios en blanco (`.text` y `.tail`) quedan descolocados.
  `ET.indent` los vuelve a calcular antes de escribir.

---
## Ejercicio 5. Lecturas de sensores con datos sucios ★★

Una estación meteorológica casera guarda sus lecturas en `datos/estacion.xml`. El fichero tiene **errores a
propósito**: decimales con coma, valores vacíos o ausentes, el texto `N/D`, espacios sobrantes, fechas en dos
formatos (`2026-10-12T08:00:00` y `12/10/2026 10:00`), una lectura sin fecha y el valor `-999`, que el sensor usa
cuando falla.

1. Escribe una función `leer_lecturas(ruta)` que devuelva una lista de lecturas válidas (diccionarios con `fecha`,
   `temperatura`, `humedad` y `presion`) y una lista de incidencias con texto legible (por ejemplo,
   `"Lectura 4: falta la temperatura"`).
2. Ningún dato erróneo debe hacer que el programa falle. Un valor no válido se guarda como `None` y genera una
   incidencia; el resto de valores de esa lectura se aprovechan.
3. Considera no válidas las temperaturas fuera de [-50, 60] y las humedades fuera de [0, 100].
4. Muestra las incidencias y la media de cada magnitud indicando cuántos valores se han usado.
5. Llama a la función con `datos/estacion_rota.xml`, que está mal formado, y con un fichero que no existe. El
   programa debe mostrar un mensaje claro: en el primer caso, la línea y la columna del error y el contenido de esa línea.

**Pista.** `ET.ParseError` tiene el atributo `position`, una tupla `(línea, columna)`.

### Solución

In [11]:
from datetime import datetime
from statistics import mean

FORMATOS_FECHA = ("%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M")
RANGOS = {"temperatura": (-50, 60), "humedad": (0, 100), "presion": (850, 1100)}


def a_numero(texto):
    """Convierte el texto de un elemento en float. Devuelve (valor, problema)."""
    if texto is None:
        return None, "falta"
    texto = texto.strip().replace(",", ".")
    if not texto:
        return None, "está vacía"
    try:
        return float(texto), None
    except ValueError:
        return None, f"tiene un valor no numérico ({texto!r})"


def a_fecha(texto):
    if not texto:
        return None, "no tiene fecha"
    for formato in FORMATOS_FECHA:
        try:
            return datetime.strptime(texto, formato), None
        except ValueError:
            pass
    return None, f"tiene una fecha con formato desconocido ({texto!r})"


def leer_lecturas(ruta):
    raiz = ET.parse(ruta).getroot()          # puede lanzar ParseError o FileNotFoundError
    lecturas, incidencias = [], []
    for n, elem in enumerate(raiz.findall("lectura"), start=1):
        lectura = {}
        lectura["fecha"], problema = a_fecha(elem.get("fecha"))
        if problema:
            incidencias.append(f"Lectura {n}: {problema}")
        for magnitud, (minimo, maximo) in RANGOS.items():
            valor, problema = a_numero(elem.findtext(magnitud))
            if problema:
                incidencias.append(f"Lectura {n}: la {magnitud} {problema}")
            elif not minimo <= valor <= maximo:
                incidencias.append(f"Lectura {n}: {magnitud} fuera de rango ({valor})")
                valor = None
            lectura[magnitud] = valor
        lecturas.append(lectura)
    return lecturas, incidencias


def informe(ruta):
    ruta = Path(ruta)
    print(f"== {ruta} ==")
    try:
        lecturas, incidencias = leer_lecturas(ruta)
    except FileNotFoundError:
        print("No existe el fichero.\n")
        return
    except ET.ParseError as e:
        linea, columna = e.position
        texto_linea = ruta.read_text(encoding="utf-8").splitlines()[linea - 1]
        print(f"XML mal formado: {e}")
        print(f"  {linea}| {texto_linea}\n")
        return

    print(f"{len(lecturas)} lecturas, {len(incidencias)} incidencias:")
    for incidencia in incidencias:
        print("  -", incidencia)
    for magnitud in RANGOS:
        valores = [l[magnitud] for l in lecturas if l[magnitud] is not None]
        if valores:
            print(f"{magnitud.capitalize():<12} media {mean(valores):7.2f} ({len(valores)} valores)")
    print()


informe(DATOS / "estacion.xml")
informe(DATOS / "estacion_rota.xml")
informe(DATOS / "no_existe.xml")

== datos/estacion.xml ==
10 lecturas, 5 incidencias:
  - Lectura 4: la temperatura falta
  - Lectura 5: la humedad está vacía
  - Lectura 6: la temperatura tiene un valor no numérico ('N/D')
  - Lectura 7: no tiene fecha
  - Lectura 9: temperatura fuera de rango (-999.0)
Temperatura  media   14.77 (7 valores)
Humedad      media   69.11 (9 valores)
Presion      media 1015.08 (10 valores)

== datos/estacion_rota.xml ==
XML mal formado: mismatched tag: line 11, column 4
  11|   </lectura>

== datos/no_existe.xml ==
No existe el fichero.



**Comentarios**

- Cada conversión devuelve el valor **y** el motivo del problema. Así el código que recorre el XML queda limpio y los
  mensajes de incidencia se construyen en un solo sitio.
- `findtext` devuelve `None` si el elemento no existe y `""` si existe vacío: son dos incidencias distintas.
- En `estacion_rota.xml` el error se detecta en `</lectura>` de la línea 11, aunque el elemento sin cerrar es
  `<humedad>` de la línea 10: el analizador solo sabe que algo va mal cuando encuentra una etiqueta de cierre que no
  corresponde. Es lo mismo que pasa con los errores de compilación. Las líneas se cuentan desde 1 y las columnas,
  desde 0.
- `-999` es un valor «centinela». Es habitual en datos de sensores y conviene comprobar rangos aunque el valor sea
  numéricamente correcto.

---
## Ejercicio 6. Ruta de senderismo en GPX ★★

GPX es el formato XML estándar para rutas GPS: lo exportan Wikiloc, Strava, Komoot, los relojes deportivos y casi
cualquier app de senderismo. En `datos/ruta.gpx` tienes una ruta de ejemplo (sintética) de subida a Peñalara. Si
tienes alguna ruta tuya, puedes usarla en su lugar.

```xml
<gpx version="1.1" xmlns="http://www.topografix.com/GPX/1/1" ...>
  <trk>
    <name>Peñalara desde Cotos</name>
    <trkseg>
      <trkpt lat="40.822600" lon="-3.958500">
        <ele>1830.0</ele>
        <time>2026-10-11T08:15:00Z</time>
      </trkpt>
      ...
```

1. Ejecuta `raiz.findall(".//trkpt")` y explica por qué devuelve una lista vacía. Muestra cómo se llama realmente la
   etiqueta (`raiz.tag`).
2. Lee todos los puntos usando un diccionario de espacios de nombres.
3. Calcula y muestra: nombre de la ruta, número de puntos, distancia total en km (fórmula del *haversine*), desnivel
   positivo y negativo, altitud máxima y mínima, duración y velocidad media.
4. Exporta los puntos a `salida/ruta.csv` con las columnas `lat`, `lon`, `ele`, `hora` y `km` (distancia acumulada).
5. *(Ampliación)* Obtén el espacio de nombres a partir de `raiz.tag` en lugar de escribirlo a mano, para que el
   programa funcione también con ficheros GPX 1.0.

**Fórmula del haversine** (distancia entre dos puntos sobre una esfera de radio R = 6371 km, con latitudes φ y
longitudes λ en radianes):

$$a = \sin^2\left(\frac{\Delta\varphi}{2}\right) + \cos\varphi_1 \cos\varphi_2 \sin^2\left(\frac{\Delta\lambda}{2}\right)
\qquad d = 2R \cdot \operatorname{atan2}\left(\sqrt{a}, \sqrt{1-a}\right)$$

### Solución

In [12]:
raiz = ET.parse(DATOS / "ruta.gpx").getroot()

print("Sin espacio de nombres:", raiz.findall(".//trkpt"))
print("Etiqueta real de la raíz:", raiz.tag)

Sin espacio de nombres: []
Etiqueta real de la raíz: {http://www.topografix.com/GPX/1/1}gpx


El atributo `xmlns="http://www.topografix.com/GPX/1/1"` declara un **espacio de nombres por defecto**: todas las
etiquetas del documento pertenecen a él aunque no lleven prefijo. ElementTree guarda el nombre completo con la
notación `{uri}etiqueta`, así que `trkpt` no coincide con `{http://www.topografix.com/GPX/1/1}trkpt`. En las
búsquedas se usa un prefijo cualquiera asociado a la URI mediante el parámetro `namespaces`.

In [13]:
import csv
import math
from datetime import datetime


def espacio_de_nombres(elemento):
    """'{uri}gpx' -> 'uri' ('' si la etiqueta no tiene espacio de nombres)."""
    return elemento.tag[1:].split("}")[0] if elemento.tag.startswith("{") else ""


def haversine(lat1, lon1, lat2, lon2, radio=6371.0):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * radio * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def leer_gpx(ruta):
    raiz = ET.parse(ruta).getroot()
    ns = {"g": espacio_de_nombres(raiz)}
    puntos = []
    for pt in raiz.findall(".//g:trkpt", ns):
        hora = pt.findtext("g:time", namespaces=ns)
        puntos.append({
            "lat": float(pt.get("lat")),
            "lon": float(pt.get("lon")),
            "ele": float(pt.findtext("g:ele", namespaces=ns)),
            "hora": datetime.fromisoformat(hora) if hora else None,
        })
    nombre = raiz.findtext("g:trk/g:name", default="(sin nombre)", namespaces=ns)
    return nombre, puntos


nombre, puntos = leer_gpx(DATOS / "ruta.gpx")

distancia = subida = bajada = 0.0
puntos[0]["km"] = 0.0
for anterior, actual in zip(puntos, puntos[1:]):
    distancia += haversine(anterior["lat"], anterior["lon"], actual["lat"], actual["lon"])
    actual["km"] = round(distancia, 3)
    desnivel = actual["ele"] - anterior["ele"]
    if desnivel > 0:
        subida += desnivel
    else:
        bajada -= desnivel

duracion = puntos[-1]["hora"] - puntos[0]["hora"]
horas = duracion.total_seconds() / 3600
altitudes = [p["ele"] for p in puntos]

print(f"Ruta: {nombre}")
print(f"Puntos: {len(puntos)}")
print(f"Distancia: {distancia:.2f} km")
print(f"Desnivel: +{subida:.0f} m / -{bajada:.0f} m")
print(f"Altitud: máx. {max(altitudes):.0f} m, mín. {min(altitudes):.0f} m")
print(f"Duración: {duracion} (velocidad media {distancia / horas:.1f} km/h)")

with open(SALIDA / "ruta.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=["lat", "lon", "ele", "hora", "km"])
    escritor.writeheader()
    escritor.writerows(puntos)
print(f"\nGuardado {SALIDA / 'ruta.csv'}")

Ruta: Peñalara desde Cotos
Puntos: 121
Distancia: 6.32 km
Desnivel: +597 m / -597 m
Altitud: máx. 2427 m, mín. 1830 m
Duración: 4:05:00 (velocidad media 1.5 km/h)

Guardado salida/ruta.csv


**Comentarios**

- `findtext` recibe los espacios de nombres como argumento con nombre (`namespaces=ns`), porque su segundo argumento
  posicional es el valor por defecto. Con `findall` y `find` sí se puede pasar en segunda posición.
- Los atributos sin prefijo, como `lat` y `lon`, **no** pertenecen al espacio de nombres por defecto: se leen con
  `get("lat")`.
- `datetime.fromisoformat` admite el sufijo `Z` desde Python 3.11.
- El desnivel acumulado de una ruta real sale más alto de lo que muestran las apps, porque el GPS tiene ruido en la
  altitud. Las apps suavizan la serie o ignoran variaciones de pocos metros.
- Para **escribir** XML con espacio de nombres sin que aparezcan prefijos `ns0:`, se registra antes con
  `ET.register_namespace("", "http://www.topografix.com/GPX/1/1")`.

---
## Ejercicio 7. Fusión de pedidos de dos sucursales ★★

Una empresa de suministros tiene dos sucursales y cada una exporta sus pedidos en un XML con estructura **plana**
(`datos/pedidos_norte.xml` y `datos/pedidos_sur.xml`):

```xml
<pedidos sucursal="Norte">
  <pedido id="1001" fecha="2026-10-01" cliente_id="C03" cliente="Ferretería Pisuerga">
    <producto>Taladro percutor</producto>
    <cantidad>2</cantidad>
    <precio_unitario>89.90</precio_unitario>
  </pedido>
```

Genera `salida/pedidos_por_cliente.xml`, con los pedidos **agrupados por cliente**:

```xml
<clientes generado="...">
  <cliente id="C01" nombre="Bricolaje Duero" num_pedidos="3" total="...">
    <pedido id="1002" sucursal="Norte" fecha="2026-10-02">
      <producto>...</producto><cantidad>...</cantidad><precio_unitario>...</precio_unitario><importe>...</importe>
    </pedido>
    ...
```

1. Los clientes se ordenan por `id` y sus pedidos, por fecha.
2. Cada pedido indica de qué sucursal viene y su importe (cantidad × precio unitario).
3. Detecta los pedidos con el mismo `id` en las dos sucursales. Si son idénticos, se incluyen una sola vez y se avisa.
   Si tienen datos distintos, se informa del conflicto y no se incluye ninguno.
4. Usa `decimal.Decimal` para el dinero.
5. Muestra un resumen por cliente.

### Solución

In [14]:
from collections import defaultdict
from datetime import datetime
from decimal import Decimal

CAMPOS = ("producto", "cantidad", "precio_unitario")


def leer_pedidos(ruta):
    raiz = ET.parse(ruta).getroot()
    sucursal = raiz.get("sucursal")
    for p in raiz.findall("pedido"):
        yield {
            "id": p.get("id"), "fecha": p.get("fecha"), "sucursal": sucursal,
            "cliente_id": p.get("cliente_id"), "cliente": p.get("cliente"),
            **{campo: p.findtext(campo) for campo in CAMPOS},
        }


def mismo_contenido(a, b):
    claves = ("fecha", "cliente_id", *CAMPOS)
    return all(a[c] == b[c] for c in claves)


# 1. Leer todos los pedidos y detectar duplicados por id
pedidos, avisos, conflictos = {}, [], set()
for ruta in (DATOS / "pedidos_norte.xml", DATOS / "pedidos_sur.xml"):
    for pedido in leer_pedidos(ruta):
        previo = pedidos.get(pedido["id"])
        if previo is None:
            pedidos[pedido["id"]] = pedido
        elif mismo_contenido(previo, pedido):
            avisos.append(f"Pedido {pedido['id']} repetido en {previo['sucursal']} y "
                          f"{pedido['sucursal']} (idéntico): se incluye una vez")
        else:
            conflictos.add(pedido["id"])
            avisos.append(f"Pedido {pedido['id']}: datos distintos en {previo['sucursal']} y "
                          f"{pedido['sucursal']}: se excluye")
for id_ in conflictos:
    del pedidos[id_]

# 2. Agrupar por cliente (estructura intermedia en Python)
por_cliente = defaultdict(list)
nombres = {}
for pedido in pedidos.values():
    por_cliente[pedido["cliente_id"]].append(pedido)
    nombres[pedido["cliente_id"]] = pedido["cliente"]

# 3. Construir el nuevo árbol
raiz = ET.Element("clientes", generado=datetime.now().isoformat(timespec="seconds"))
for cliente_id in sorted(por_cliente):
    lista = sorted(por_cliente[cliente_id], key=lambda p: (p["fecha"], p["id"]))
    cliente = ET.SubElement(raiz, "cliente", id=cliente_id, nombre=nombres[cliente_id],
                            num_pedidos=str(len(lista)))
    total = Decimal("0")
    for p in lista:
        importe = int(p["cantidad"]) * Decimal(p["precio_unitario"])
        total += importe
        elem = ET.SubElement(cliente, "pedido", id=p["id"], sucursal=p["sucursal"], fecha=p["fecha"])
        for campo in CAMPOS:
            ET.SubElement(elem, campo).text = p[campo]
        ET.SubElement(elem, "importe").text = f"{importe:.2f}"
    cliente.set("total", f"{total:.2f}")

arbol = ET.ElementTree(raiz)
ET.indent(arbol)
arbol.write(SALIDA / "pedidos_por_cliente.xml", encoding="utf-8", xml_declaration=True)

# 4. Resumen
for aviso in avisos:
    print("Aviso:", aviso)
print()
for c in raiz.findall("cliente"):
    print(f"{c.get('id')} {c.get('nombre'):<24} {c.get('num_pedidos')} pedidos {c.get('total'):>9} €")

Aviso: Pedido 1003 repetido en Norte y Sur (idéntico): se incluye una vez

C01 Bricolaje Duero          3 pedidos    379.45 €
C02 Construcciones Arlanza   2 pedidos   1239.00 €
C03 Ferretería Pisuerga      2 pedidos    308.80 €
C04 Reformas Tormes          2 pedidos    245.00 €


**Comentarios**

- Transformar una jerarquía en otra se hace en dos fases: **leer** a estructuras de Python (diccionarios y listas) y
  **construir** después el árbol nuevo. Intentar mover elementos directamente de un árbol a otro es más difícil de
  seguir y obliga a modificar los originales.
- `Decimal("89.90")` se crea a partir del **texto**, nunca de un `float`, para no arrastrar errores de redondeo.
- Las fechas en formato ISO (`AAAA-MM-DD`) se ordenan correctamente como cadenas. Con `dd/mm/aaaa` habría que
  convertirlas antes.

---
## Ejercicio 8. Mirar dentro de un `.docx` ★★★

Un documento de Word (`.docx`), igual que un `.xlsx` o un `.odt`, es un **fichero ZIP** que contiene varios XML.
El texto está en `word/document.xml`, los estilos en `word/styles.xml` y los metadatos en `docProps/core.xml`.

Trabaja con `datos/memoria.docx` usando solo `zipfile` y ElementTree:

1. Lista los ficheros que contiene el ZIP.
2. Muestra el título y el autor del documento y su fecha de creación, que están en `docProps/core.xml`.
3. Cuenta los párrafos con texto y las palabras.
4. Muestra el índice del documento: los títulos con sangría según su nivel.
5. Guarda el texto plano en `salida/memoria.txt`, con una línea en blanco entre párrafos.

**Lo que necesitas saber del formato.** Cada párrafo es un `<w:p>`. Dentro, el texto está repartido en uno o varios
fragmentos `<w:r>` (*runs*), cada uno con su `<w:t>`: una palabra en negrita en mitad de una frase parte el texto
en tres *runs*. El estilo del párrafo está en `<w:pPr><w:pStyle w:val="Heading1"/></w:pPr>`.

**Cuidado con los estilos.** El identificador del estilo depende del idioma de Word: en un Word en español el
título 1 se llama `Ttulo1`. Para que el programa sirva con cualquier documento, busca en `word/styles.xml` el
nombre interno del estilo (`<w:name w:val="heading 1"/>`), que siempre está en inglés.

```python
NS = {
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "cp": "http://schemas.openxmlformats.org/package/2006/metadata/core-properties",
}
```

### Solución

In [15]:
import zipfile

NS = {
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "cp": "http://schemas.openxmlformats.org/package/2006/metadata/core-properties",
}
W = "{" + NS["w"] + "}"            # para leer atributos con espacio de nombres: W + "val"


def niveles_de_titulo(styles_xml):
    """styleId -> nivel (0 = título del documento, 1 = heading 1...)."""
    niveles = {}
    for estilo in ET.fromstring(styles_xml).findall("w:style[@w:type='paragraph']", NS):
        nombre = estilo.find("w:name", NS).get(W + "val").lower()
        if nombre == "title":
            niveles[estilo.get(W + "styleId")] = 0
        elif nombre.startswith("heading "):
            niveles[estilo.get(W + "styleId")] = int(nombre.split()[1])
    return niveles


def texto_parrafo(p):
    return "".join(t.text or "" for t in p.iter(W + "t"))


with zipfile.ZipFile(DATOS / "memoria.docx") as docx:
    print("Contenido del ZIP:")
    for nombre in docx.namelist():
        print("  ", nombre)
    core = ET.fromstring(docx.read("docProps/core.xml"))
    niveles = niveles_de_titulo(docx.read("word/styles.xml"))
    documento = ET.fromstring(docx.read("word/document.xml"))

print("\nTítulo:", core.findtext("dc:title", namespaces=NS) or "(sin título)")
print("Autor: ", core.findtext("dc:creator", namespaces=NS))
print("Creado:", core.findtext("dcterms:created", namespaces=NS))

parrafos = []
indice = []
for p in documento.findall("w:body/w:p", NS):
    texto = texto_parrafo(p)
    if not texto.strip():
        continue
    parrafos.append(texto)
    estilo = p.find("w:pPr/w:pStyle", NS)
    if estilo is not None and estilo.get(W + "val") in niveles:
        indice.append((niveles[estilo.get(W + "val")], texto))

palabras = sum(len(t.split()) for t in parrafos)
print(f"\n{len(parrafos)} párrafos con texto, {palabras} palabras\n")

print("Índice:")
for nivel, texto in indice:
    print("  " * nivel + texto)

(SALIDA / "memoria.txt").write_text("\n\n".join(parrafos) + "\n", encoding="utf-8")
print(f"\nTexto guardado en {SALIDA / 'memoria.txt'}")

Contenido del ZIP:
   [Content_Types].xml
   _rels/.rels
   docProps/core.xml
   docProps/app.xml
   word/document.xml
   word/_rels/document.xml.rels
   word/styles.xml
   word/stylesWithEffects.xml
   word/settings.xml
   word/webSettings.xml
   word/fontTable.xml
   word/theme/theme1.xml
   customXml/item1.xml
   customXml/_rels/item1.xml.rels
   customXml/itemProps1.xml
   word/numbering.xml
   docProps/thumbnail.jpeg

Título: (sin título)
Autor:  python-docx
Creado: 2013-12-23T23:15:00Z

16 párrafos con texto, 156 palabras

Índice:
Memoria del proyecto intermodular
  1. Introducción
  2. Análisis
    2.1. Requisitos funcionales
    2.2. Requisitos no funcionales
  3. Diseño de datos
  4. Conclusiones

Texto guardado en salida/memoria.txt


In [16]:
# El párrafo de la introducción está partido en varios runs por la negrita
p = documento.findall("w:body/w:p", NS)[3]
for i, run in enumerate(p.findall("w:r", NS), start=1):
    negrita = run.find("w:rPr/w:b", NS) is not None
    print(f"run {i}{' (negrita)' if negrita else ''}: {run.findtext('w:t', namespaces=NS)!r}")

run 1: 'El objetivo del proyecto es desarrollar una aplicación para gestionar '
run 2 (negrita): 'las reservas de las pistas deportivas'
run 3: ' del ayuntamiento, sustituyendo el sistema actual basado en hojas de cálculo.'


**Comentarios**

- Los **atributos** con prefijo, como `w:val`, también llevan el espacio de nombres: se leen con
  `get("{uri}val")`. El diccionario `namespaces` solo sirve para las rutas de `find` y `findall`, no para `get`.
  Sí se puede usar el prefijo dentro de un predicado: `w:style[@w:type='paragraph']`.
- Este documento lo ha creado la biblioteca `python-docx`, que pone su propio nombre como autor y deja el título
  vacío; la fecha de creación es la de la plantilla interna de la biblioteca. Un documento de Word real tendría el
  nombre del usuario y la fecha en que se creó.
- La lista con viñetas de los requisitos usa el estilo «List Bullet»: no es un título, así que no aparece en el índice.

---
## Ejercicio 9. Un fichero XML enorme ★★★

`ET.parse` construye **todo el árbol en memoria** antes de devolverlo. Con ficheros de cientos de MB o varios GB,
esto puede agotar la memoria. `ET.iterparse` lee el fichero poco a poco y avisa cada vez que se abre o se cierra un
elemento, de modo que se puede procesar cada uno y liberarlo enseguida, como se hace con SAX.

1. Ejecuta la celda que genera `salida/accesos.xml`, un registro de peticiones a un servidor web:
   ```xml
   <peticion ip="10.0.3.17" fecha="2026-10-12T14:03:22" metodo="GET" codigo="200" bytes="5321">
     <ruta>/productos/23</ruta>
     <agente>Mozilla/5.0 ...</agente>
   </peticion>
   ```
   Por defecto tiene 200 000 peticiones (unos 40 MB). Si tu ordenador lo aguanta, prueba con 2 000 000 (unos 400 MB).
2. Escribe `estadisticas_parse(ruta)`, que calcule con `ET.parse`: número de peticiones por código de estado, la IP con
   más peticiones, la hora del día con más tráfico y el total de bytes servidos.
3. Escribe `estadisticas_iterparse(ruta)`, que calcule lo mismo con `ET.iterparse` y libere cada `<peticion>` en
   cuanto la haya procesado.
4. Mide el pico de memoria (`tracemalloc`) y el tiempo (`time.perf_counter`) de cada versión, comprueba que dan el
   mismo resultado y muestra una tabla comparativa.

**Pista.** `elem.clear()` vacía el elemento, pero el elemento vacío sigue colgando de la raíz. Para liberar de
verdad la memoria hay que quitarlo también de su padre, o vaciar la raíz cada cierto tiempo. Para tener una referencia
a la raíz, pide también los eventos `"start"`.

In [17]:
import random
from datetime import datetime, timedelta


def generar_registro(ruta, n=200_000, semilla=1):
    """Escribe el XML línea a línea, sin construir el árbol: así no ocupa memoria."""
    rnd = random.Random(semilla)
    ips = [f"10.0.{rnd.randint(0, 9)}.{rnd.randint(1, 254)}" for _ in range(400)]
    rutas = ["/", "/productos", "/carrito", "/login", "/api/stock"] + [f"/productos/{i}" for i in range(1, 60)]
    agentes = ["Mozilla/5.0 (Windows NT 10.0; Win64; x64)", "Mozilla/5.0 (X11; Linux x86_64)",
               "Mozilla/5.0 (iPhone; CPU iPhone OS 18_0 like Mac OS X)", "curl/8.9.1"]
    codigos = ["200"] * 85 + ["304"] * 6 + ["404"] * 6 + ["500"] * 2 + ["403"]
    pesos_hora = [1, 1, 1, 1, 1, 2, 3, 5, 8, 9, 10, 11, 12, 10, 9, 9, 10, 12, 14, 13, 11, 8, 5, 3]
    inicio = datetime(2026, 10, 12)
    with open(ruta, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n<registro servidor="web01">\n')
        for _ in range(n):
            hora = rnd.choices(range(24), weights=pesos_hora)[0]
            fecha = inicio + timedelta(hours=hora, seconds=rnd.randrange(3600))
            f.write(f'  <peticion ip="{rnd.choice(ips)}" fecha="{fecha:%Y-%m-%dT%H:%M:%S}" '
                    f'metodo="{rnd.choice(["GET"] * 9 + ["POST"])}" codigo="{rnd.choice(codigos)}" '
                    f'bytes="{rnd.randint(200, 90_000)}">\n'
                    f'    <ruta>{rnd.choice(rutas)}</ruta>\n'
                    f'    <agente>{rnd.choice(agentes)}</agente>\n'
                    f'  </peticion>\n')
        f.write("</registro>\n")


ACCESOS = SALIDA / "accesos.xml"
generar_registro(ACCESOS)
print(f"{ACCESOS}: {ACCESOS.stat().st_size / 1_000_000:.1f} MB")

salida/accesos.xml: 39.4 MB


### Solución

In [18]:
import time
import tracemalloc
from collections import Counter


class Estadisticas:
    def __init__(self):
        self.codigos = Counter()
        self.ips = Counter()
        self.horas = Counter()
        self.bytes = 0

    def anotar(self, peticion):
        self.codigos[peticion.get("codigo")] += 1
        self.ips[peticion.get("ip")] += 1
        self.horas[peticion.get("fecha")[11:13]] += 1
        self.bytes += int(peticion.get("bytes"))

    def resumen(self):
        ip, n_ip = self.ips.most_common(1)[0]
        hora, n_hora = self.horas.most_common(1)[0]
        return {"peticiones": sum(self.codigos.values()), "codigos": dict(sorted(self.codigos.items())),
                "ip_mas_activa": f"{ip} ({n_ip})", "hora_punta": f"{hora}h ({n_hora})",
                "GB_servidos": round(self.bytes / 1e9, 3)}


def estadisticas_parse(ruta):
    est = Estadisticas()
    for peticion in ET.parse(ruta).getroot().iter("peticion"):
        est.anotar(peticion)
    return est.resumen()


def estadisticas_iterparse(ruta):
    est = Estadisticas()
    raiz = None
    for evento, elem in ET.iterparse(ruta, events=("start", "end")):
        if evento == "start":
            if raiz is None:
                raiz = elem                     # el primer "start" es el de la raíz
        elif elem.tag == "peticion":            # evento "end": el elemento ya está completo
            est.anotar(elem)
            raiz.clear()                        # suelta las peticiones ya procesadas
    return est.resumen()


def medir(funcion, ruta):
    tracemalloc.start()
    t0 = time.perf_counter()
    resultado = funcion(ruta)
    segundos = time.perf_counter() - t0
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return resultado, segundos, pico / 1_000_000


r1, t1, m1 = medir(estadisticas_parse, ACCESOS)
r2, t2, m2 = medir(estadisticas_iterparse, ACCESOS)

assert r1 == r2, "Las dos versiones deberían dar el mismo resultado"
for clave, valor in r1.items():
    print(f"{clave:<14} {valor}")

print(f"\n{'Método':<12} {'Tiempo (s)':>11} {'Pico de memoria (MB)':>22}")
print(f"{'parse':<12} {t1:>11.2f} {m1:>22.1f}")
print(f"{'iterparse':<12} {t2:>11.2f} {m2:>22.1f}")

peticiones     200000
codigos        {'200': 170055, '304': 11885, '403': 2068, '404': 11965, '500': 4027}
ip_mas_activa  10.0.0.102 (1513)
hora_punta     18h (16562)
GB_servidos    9.015

Método        Tiempo (s)   Pico de memoria (MB)
parse              20.34                  262.0
iterparse           4.53                    0.2


**Comentarios**

- Con `parse`, el pico de memoria crece con el tamaño del fichero (varias veces su tamaño en disco, porque cada
  elemento es un objeto de Python). Con `iterparse` y `raiz.clear()` se mantiene casi constante: duplicar el fichero
  apenas cambia la memoria.
- `tracemalloc` hace que todo vaya más lento mientras mide. Los tiempos sirven para comparar los dos métodos entre sí,
  no como valores absolutos.
- En el evento `"start"` los atributos ya están disponibles, pero el texto y los hijos todavía no: por eso las
  estadísticas se anotan en el evento `"end"`.
- `elem.clear()` por sí solo no basta: deja miles de elementos vacíos colgando de la raíz. Vaciar la raíz entera
  funciona aquí porque no necesitamos nada de lo ya procesado.
- El fichero generado ocupa bastante: bórralo de `salida/` cuando termines.

---
## Ejercicio 10. Proyecto: el tiempo en Castilla y León con AEMET ★★★

AEMET publica en su web la predicción de los próximos siete días para cada municipio en XML. No hace falta
registrarse ni pedir una clave:

```
https://www.aemet.es/xml/municipios/localidad_<código>.xml
```

El código es el del INE, con cinco cifras. Es el número que aparece al final de la ficha del municipio en aemet.es:
`.../prediccion/municipios/valladolid-id47186` → `47186`.

> **Uso responsable.** Guarda las descargas en una caché, no hagas peticiones en bucle y deja una pausa entre
> municipios. El aviso legal de AEMET exige citarla como fuente de los datos.

En `datos/aemet/localidad_47186.xml` tienes una copia de la predicción real de Valladolid del 19/10/2026 para poder
trabajar sin conexión. Ábrela y fíjate en estos detalles antes de empezar:

- La codificación es **ISO-8859-15**, no UTF-8.
- Hay muchos elementos vacíos, como `<prob_precipitacion periodo="00-06"/>` o `<racha_max/>`.
- Los dos primeros días tienen datos por periodos de 6 horas (`00-06`, `06-12`…), los dos siguientes por periodos
  de 12 horas y los tres últimos no tienen atributo `periodo`.
- En el primer día, los periodos que ya han pasado están vacíos, incluido el del día completo (`00-24`).
- Algunos códigos de estado del cielo llevan una `n` de «noche», como `81n`.

### Apartados

1. **Descarga con caché.** Escribe `obtener_xml(codigo)`, que devuelva los **bytes** del XML. Si el fichero de caché
   tiene menos de una hora, lo usa; si no, lo descarga y lo guarda. Si la descarga falla y hay una copia antigua, la
   usa avisando.
2. **Codificación.** Comprueba qué pasa si decodificas los bytes como UTF-8 y explica por qué conviene pasar los
   bytes directamente a `ET.fromstring`.
3. **Modelo.** Convierte el XML en objetos: una *dataclass* `Prediccion` (código, nombre, provincia, fecha de
   elaboración y lista de días) y otra `DiaPrediccion` (fecha, máxima, mínima, probabilidad de precipitación del
   día, descripción y código del cielo, racha máxima). Los datos ausentes se guardan como `None`. Decide y justifica
   cómo obtienes el dato «del día completo» cuando `00-24` está vacío.
4. **Varias capitales.** Procesa las nueve capitales de provincia de Castilla y León y muestra una tabla con la
   predicción para **mañana** (el día siguiente a la fecha de elaboración), la capital más cálida y la más lluviosa.
5. **Resumen propio.** Genera `salida/resumen_cyl.xml` con una estructura diseñada por ti y expórtalo también a
   `salida/resumen_cyl.json` y `salida/resumen_cyl.csv`.
6. *(Ampliación)* **Histórico.** Cada vez que se ejecute, añade la predicción a `salida/historico.xml` (sin duplicar
   una misma elaboración). Escribe una función que muestre todas las predicciones guardadas para una fecha y
   municipio, para ver cómo cambia la predicción según se acerca el día.
7. *(Ampliación avanzada)* **Avisos.** Con una clave gratuita de AEMET OpenData, descarga los avisos en formato CAP de
   Castilla y León, descomprímelos con `tarfile` y lista los avisos activos.

### Solución

#### 1. Descarga con caché

In [19]:
import time
import urllib.error
import urllib.request
from datetime import datetime, timedelta

URL_AEMET = "https://www.aemet.es/xml/municipios/localidad_{codigo}.xml"
CACHE = DATOS / "aemet"
CACHE.mkdir(exist_ok=True)

CAPITALES_CYL = {
    "05019": "Ávila", "09059": "Burgos", "24089": "León",
    "34120": "Palencia", "37274": "Salamanca", "40194": "Segovia",
    "42173": "Soria", "47186": "Valladolid", "49275": "Zamora",
}


def obtener_xml(codigo, max_edad=timedelta(hours=1)):
    """Devuelve (bytes del XML, origen). Lanza OSError si no hay ni conexión ni copia."""
    fichero = CACHE / f"localidad_{codigo}.xml"
    if fichero.exists():
        edad = datetime.now() - datetime.fromtimestamp(fichero.stat().st_mtime)
        if edad < max_edad:
            return fichero.read_bytes(), "caché"
    try:
        peticion = urllib.request.Request(
            URL_AEMET.format(codigo=codigo),
            headers={"User-Agent": "Ejercicios-AccesoDatos/1.0 (uso educativo)"})
        with urllib.request.urlopen(peticion, timeout=10) as respuesta:
            contenido = respuesta.read()
        ET.fromstring(contenido)              # no guardar nada que no sea XML bien formado
        fichero.write_bytes(contenido)
        time.sleep(1)                         # pausa entre peticiones
        return contenido, "AEMET"
    except (OSError, ET.ParseError) as e:     # URLError y TimeoutError heredan de OSError
        if fichero.exists():
            return fichero.read_bytes(), f"copia antigua (sin conexión: {e.__class__.__name__})"
        raise OSError(f"No se pudo descargar {codigo} y no hay copia en caché") from e


contenido, origen = obtener_xml("47186")
print(f"{len(contenido)} bytes obtenidos de: {origen}")

11920 bytes obtenidos de: caché


#### 2. Codificación

In [20]:
print(contenido[:60])
try:
    contenido.decode("utf-8")
except UnicodeDecodeError as e:
    print("Error al decodificar como UTF-8:", e)

raiz = ET.fromstring(contenido)              # los bytes: ElementTree lee la declaración
print(raiz.findtext("origen/productor"))

b"<?xml version='1.0' encoding='ISO-8859-15'?>\n<root xmlns:xsd"
Error al decodificar como UTF-8: 'utf-8' codec can't decode byte 0xed in position 303: invalid continuation byte
Agencia Estatal de Meteorología - AEMET. Gobierno de España


El fichero declara `encoding='ISO-8859-15'` en su primera línea. Si se pasan los **bytes** a `ET.fromstring`, el
analizador lee esa declaración y decodifica bien las tildes y la «ñ». Si los decodificamos nosotros con la
codificación equivocada, falla (como con UTF-8) o, peor aún, produce caracteres extraños sin avisar. Con `requests`
pasa lo mismo: hay que usar `respuesta.content` (bytes), no `respuesta.text`, que adivina la codificación a
partir de las cabeceras HTTP.

#### 3. Modelo

In [21]:
from dataclasses import dataclass, field, asdict
from datetime import date

# Orden de preferencia para elegir el dato "del día": primero el día completo (00-24) o sin periodo,
# después las partes del día de más larga a más corta, dando prioridad a la tarde.
PREFERENCIA = ["00-24", None, "12-24", "00-12", "12-18", "06-12", "18-24", "00-06"]


@dataclass
class DiaPrediccion:
    fecha: date
    maxima: int | None
    minima: int | None
    prob_precipitacion: int | None
    cielo: str | None
    codigo_cielo: str | None
    racha_max: int | None


@dataclass
class Prediccion:
    codigo: str
    nombre: str
    provincia: str
    elaborado: datetime
    dias: list[DiaPrediccion] = field(default_factory=list)

    def dia(self, fecha):
        return next((d for d in self.dias if d.fecha == fecha), None)


def entero(texto):
    """'15' -> 15; None, '' o '  ' -> None."""
    return int(texto) if texto and texto.strip() else None


def interpretar_codigo_cielo(codigo):
    """'81n' -> (81, True); '13' -> (13, False)."""
    nocturno = codigo.endswith("n")
    return int(codigo.rstrip("n")), nocturno


def por_periodo(dia, etiqueta):
    """Elementos con datos ordenados según PREFERENCIA."""
    con_datos = [e for e in dia.findall(etiqueta) if (e.text or "").strip()]
    return sorted(con_datos, key=lambda e: PREFERENCIA.index(e.get("periodo")))


def leer_dia(dia):
    # Probabilidad: la del día completo si existe; si no, la mayor de los periodos con datos
    probs = por_periodo(dia, "prob_precipitacion")
    if probs and probs[0].get("periodo") in ("00-24", None):
        prob = int(probs[0].text)
    else:
        prob = max((int(e.text) for e in probs), default=None)

    cielos = por_periodo(dia, "estado_cielo")
    cielo = cielos[0] if cielos else None

    rachas = [int(e.text) for e in por_periodo(dia, "racha_max")]

    return DiaPrediccion(
        fecha=date.fromisoformat(dia.get("fecha")),
        maxima=entero(dia.findtext("temperatura/maxima")),
        minima=entero(dia.findtext("temperatura/minima")),
        prob_precipitacion=prob,
        cielo=cielo.get("descripcion").strip() if cielo is not None else None,
        codigo_cielo=cielo.text.strip() if cielo is not None else None,
        racha_max=max(rachas, default=None),
    )


def leer_prediccion(contenido):
    raiz = ET.fromstring(contenido)
    return Prediccion(
        codigo=raiz.get("id"),
        nombre=raiz.findtext("nombre"),
        provincia=raiz.findtext("provincia"),
        elaborado=datetime.fromisoformat(raiz.findtext("elaborado")),
        dias=[leer_dia(d) for d in raiz.findall("prediccion/dia")],
    )


def txt(valor):
    """Para mostrar datos que pueden faltar."""
    return "-" if valor is None else str(valor)


valladolid = leer_prediccion(contenido)
print(f"{valladolid.nombre} ({valladolid.provincia}), elaborado {valladolid.elaborado:%d/%m/%Y %H:%M}")
for d in valladolid.dias:
    codigo = txt(d.codigo_cielo)
    if d.codigo_cielo and interpretar_codigo_cielo(d.codigo_cielo)[1]:
        codigo += " (noche)"
    print(f"  {d.fecha:%d/%m}  {txt(d.minima):>3}/{txt(d.maxima):<3}°C  "
          f"lluvia {txt(d.prob_precipitacion):>3}%  {txt(d.cielo):<20} código {codigo}"
          + (f"  racha {d.racha_max} km/h" if d.racha_max else ""))

Valladolid (Valladolid), elaborado 19/10/2026 11:41
  19/10    6/15 °C  lluvia 100%  Cubierto             código 16
  20/10    3/16 °C  lluvia   0%  Muy nuboso           código 15
  21/10    6/16 °C  lluvia   0%  Poco nuboso          código 12
  22/10    4/17 °C  lluvia   5%  Nubes altas          código 17
  23/10    7/18 °C  lluvia  15%  Intervalos nubosos   código 13
  24/10    5/17 °C  lluvia  30%  Intervalos nubosos   código 13
  25/10   12/22 °C  lluvia  40%  Muy nuboso           código 15  racha 50 km/h


**Decisiones del modelo**

- **Probabilidad de precipitación.** Si no hay dato para el día completo, se toma la **mayor** de los periodos con
  dato: si a mediodía hay un 100 % de probabilidad de lluvia, la del día no puede ser menor. En el primer día solo
  quedan los periodos que aún no han pasado.
- **Estado del cielo.** No se puede «sumar», así que se elige el del periodo más representativo según `PREFERENCIA`.
- **Códigos del cielo.** Se guardan como texto (`"81n"`) para no perder información, y `interpretar_codigo_cielo`
  separa el número y la marca de noche cuando hace falta operar con ellos.

#### 4. Las nueve capitales

In [22]:
predicciones = []
for codigo, nombre in CAPITALES_CYL.items():
    try:
        datos_xml, origen = obtener_xml(codigo)
    except OSError as e:
        print(f"  {nombre}: {e}")
        continue
    predicciones.append(leer_prediccion(datos_xml))
    print(f"  {nombre}: {origen}")

filas = []
for p in predicciones:
    manana = p.dia(p.elaborado.date() + timedelta(days=1))
    if manana is not None:
        filas.append((p, manana))

if filas:
    print(f"\nPredicción para mañana ({filas[0][1].fecha:%d/%m/%Y})")
    print(f"{'Capital':<12} {'Mín':>4} {'Máx':>4} {'Lluvia':>7}  Cielo")
    for p, d in sorted(filas, key=lambda f: f[0].nombre):
        print(f"{p.nombre:<12} {txt(d.minima):>4} {txt(d.maxima):>4} "
              f"{txt(d.prob_precipitacion):>6}%  {txt(d.cielo)}")
    calida = max(filas, key=lambda f: f[1].maxima if f[1].maxima is not None else -99)
    lluviosa = max(filas, key=lambda f: f[1].prob_precipitacion or 0)
    print(f"\nMás cálida: {calida[0].nombre} ({calida[1].maxima} °C)")
    print(f"Más lluviosa: {lluviosa[0].nombre} ({lluviosa[1].prob_precipitacion} %)")
print("\nFuente: © AEMET")

  Ávila: No se pudo descargar 05019 y no hay copia en caché


  Burgos: No se pudo descargar 09059 y no hay copia en caché


  León: No se pudo descargar 24089 y no hay copia en caché


  Palencia: No se pudo descargar 34120 y no hay copia en caché


  Salamanca: No se pudo descargar 37274 y no hay copia en caché


  Segovia: No se pudo descargar 40194 y no hay copia en caché


  Soria: No se pudo descargar 42173 y no hay copia en caché
  Valladolid: caché


  Zamora: No se pudo descargar 49275 y no hay copia en caché

Predicción para mañana (20/10/2026)
Capital       Mín  Máx  Lluvia  Cielo
Valladolid      3   16      0%  Muy nuboso

Más cálida: Valladolid (16 °C)
Más lluviosa: Valladolid (0 %)

Fuente: © AEMET


Sin conexión solo aparece Valladolid, que es la única capital con copia en `datos/aemet/`. Con conexión se descargan
las nueve, con una pausa de un segundo entre peticiones, y la segunda ejecución en menos de una hora ya no hace
ninguna petición.

#### 5. Resumen en XML, JSON y CSV

In [23]:
import csv
import json


def resumen_a_xml(predicciones):
    raiz = ET.Element("resumen_prediccion", fuente="AEMET",
                      generado=datetime.now().isoformat(timespec="seconds"))
    for p in predicciones:
        municipio = ET.SubElement(raiz, "municipio", codigo=p.codigo, nombre=p.nombre,
                                  provincia=p.provincia, elaborado=p.elaborado.isoformat())
        for d in p.dias:
            dia = ET.SubElement(municipio, "dia", fecha=d.fecha.isoformat())
            if d.cielo:
                ET.SubElement(dia, "cielo", codigo=d.codigo_cielo).text = d.cielo
            temperatura = ET.SubElement(dia, "temperatura", unidad="C")
            for nombre in ("minima", "maxima"):
                valor = getattr(d, nombre)
                if valor is not None:
                    temperatura.set(nombre, str(valor))
            if d.prob_precipitacion is not None:
                ET.SubElement(dia, "precipitacion", probabilidad=str(d.prob_precipitacion))
            if d.racha_max is not None:
                ET.SubElement(dia, "racha_max", unidad="km/h").text = str(d.racha_max)
    arbol = ET.ElementTree(raiz)
    ET.indent(arbol)
    return arbol


resumen_a_xml(predicciones).write(SALIDA / "resumen_cyl.xml", encoding="utf-8", xml_declaration=True)

with open(SALIDA / "resumen_cyl.json", "w", encoding="utf-8") as f:
    json.dump([asdict(p) for p in predicciones], f, ensure_ascii=False, indent=2, default=str)

with open(SALIDA / "resumen_cyl.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.writer(f)
    escritor.writerow(["codigo", "municipio", "fecha", "minima", "maxima",
                       "prob_precipitacion", "cielo", "racha_max"])
    for p in predicciones:
        for d in p.dias:
            escritor.writerow([p.codigo, p.nombre, d.fecha, d.minima, d.maxima,
                               d.prob_precipitacion, d.cielo, d.racha_max])

print((SALIDA / "resumen_cyl.xml").read_text(encoding="utf-8")[:900], "...")

<?xml version='1.0' encoding='utf-8'?>
<resumen_prediccion fuente="AEMET" generado="2026-10-19T15:17:10">
  <municipio codigo="47186" nombre="Valladolid" provincia="Valladolid" elaborado="2026-10-19T11:41:07">
    <dia fecha="2026-10-19">
      <cielo codigo="16">Cubierto</cielo>
      <temperatura unidad="C" minima="6" maxima="15" />
      <precipitacion probabilidad="100" />
    </dia>
    <dia fecha="2026-10-20">
      <cielo codigo="15">Muy nuboso</cielo>
      <temperatura unidad="C" minima="3" maxima="16" />
      <precipitacion probabilidad="0" />
    </dia>
    <dia fecha="2026-10-21">
      <cielo codigo="12">Poco nuboso</cielo>
      <temperatura unidad="C" minima="6" maxima="16" />
      <precipitacion probabilidad="0" />
    </dia>
    <dia fecha="2026-10-22">
      <cielo codigo="17">Nubes altas</cielo>
      <temperatura unidad="C" minima="4" maxima="17" />
      <precipita ...


En el resumen, los datos ausentes **no** se escriben: un elemento o atributo que no existe se distingue mejor de un
cero que un elemento vacío. `asdict` convierte las *dataclasses* anidadas en diccionarios, y `default=str` resuelve
las fechas al pasar a JSON.

#### 6. Ampliación: histórico

In [24]:
HISTORICO = SALIDA / "historico.xml"


def guardar_en_historico(pred, ruta=HISTORICO):
    if ruta.exists():
        arbol = ET.parse(ruta)
    else:
        arbol = ET.ElementTree(ET.Element("historico", fuente="AEMET"))
    raiz = arbol.getroot()

    municipio = raiz.find(f"municipio[@codigo='{pred.codigo}']")
    if municipio is None:
        municipio = ET.SubElement(raiz, "municipio", codigo=pred.codigo, nombre=pred.nombre)

    elaborado = pred.elaborado.isoformat()
    if municipio.find(f"prediccion[@elaborado='{elaborado}']") is not None:
        return False                                   # ya estaba guardada

    snapshot = ET.SubElement(municipio, "prediccion", elaborado=elaborado)
    for d in pred.dias:
        atributos = {"fecha": d.fecha.isoformat()}
        for campo in ("minima", "maxima", "prob_precipitacion"):
            if getattr(d, campo) is not None:
                atributos[campo] = str(getattr(d, campo))
        ET.SubElement(snapshot, "dia", atributos)

    ET.indent(arbol)
    arbol.write(ruta, encoding="utf-8", xml_declaration=True)
    return True


def evolucion(codigo, fecha, ruta=HISTORICO):
    """Todas las predicciones guardadas para un municipio y una fecha."""
    raiz = ET.parse(ruta).getroot()
    print(f"Predicciones para el {fecha} en {codigo}:")
    for snapshot in raiz.findall(f"municipio[@codigo='{codigo}']/prediccion"):
        dia = snapshot.find(f"dia[@fecha='{fecha}']")
        if dia is not None:
            print(f"  elaborada {snapshot.get('elaborado')}: {dia.get('minima', '?')}/"
                  f"{dia.get('maxima', '?')} °C, lluvia {dia.get('prob_precipitacion', '?')} %")


for p in predicciones:
    nueva = guardar_en_historico(p)
    print(f"{p.nombre}: {'guardada' if nueva else 'ya estaba en el histórico'}")

# Por ejemplo, el cuarto día de la predicción de Valladolid
evolucion("47186", valladolid.dias[3].fecha.isoformat())

Valladolid: guardada
Predicciones para el 2026-10-22 en 47186:
  elaborada 2026-10-19T11:41:07: 4/17 °C, lluvia 5 %


Con una sola descarga solo hay una fila. Ejecuta el cuaderno varios días seguidos (o crea una tarea programada) y
verás cómo cambia la predicción para una misma fecha.

#### 7. Ampliación avanzada: avisos CAP

Este apartado necesita una clave de AEMET OpenData, que se pide gratis en
<https://opendata.aemet.es/centrodedescargas/inicio>. **El código siguiente no se ha ejecutado** al preparar el
cuaderno, porque requiere la clave; tómalo como guía y comprueba en la documentación de la API el endpoint de avisos
y el código de área de Castilla y León.

La API de OpenData funciona en dos pasos: la primera petición devuelve un JSON con una URL en el campo `datos`, y es
en esa segunda URL donde están los datos. En el caso de los avisos, un fichero `tar` con un XML en formato CAP
(*Common Alerting Protocol*) por aviso. CAP usa el espacio de nombres `urn:oasis:names:tc:emergency:cap:1.2`.

```python
import io
import json
import tarfile

API_KEY = "tu-clave"
AREA = "..."   # código de área de Castilla y León (consúltalo en la documentación)
url = f"https://opendata.aemet.es/opendata/api/avisos_cap/ultimoelaborado/area/{AREA}"

peticion = urllib.request.Request(url, headers={"api_key": API_KEY})
with urllib.request.urlopen(peticion, timeout=15) as r:
    meta = json.load(r)                                # {"estado": 200, "datos": "https://..."}
with urllib.request.urlopen(meta["datos"], timeout=15) as r:
    paquete = r.read()

CAP = {"cap": "urn:oasis:names:tc:emergency:cap:1.2"}
with tarfile.open(fileobj=io.BytesIO(paquete)) as tar:   # detecta si está comprimido
    for miembro in tar.getmembers():
        if not miembro.isfile():
            continue
        alerta = ET.fromstring(tar.extractfile(miembro).read())
        for info in alerta.findall("cap:info", CAP):
            if info.findtext("cap:language", namespaces=CAP) != "es-ES":
                continue                                   # cada aviso viene en varios idiomas
            print(info.findtext("cap:event", namespaces=CAP),
                  info.findtext("cap:severity", namespaces=CAP),
                  info.findtext("cap:onset", namespaces=CAP), "→",
                  info.findtext("cap:expires", namespaces=CAP),
                  "|", info.findtext("cap:area/cap:areaDesc", namespaces=CAP))
```

---
### Fuentes

- [Documentación de `xml.etree.ElementTree`](https://docs.python.org/3/library/xml.etree.elementtree.html)
- [Predicción por municipios de AEMET en XML](https://www.aemet.es/xml/municipios/localidad_47186.xml) y
  [AEMET OpenData](https://opendata.aemet.es/). Datos meteorológicos: © AEMET.
- [Formato GPX 1.1](https://www.topografix.com/GPX/1/1/)
- [Especificación CAP 1.2 (OASIS)](https://docs.oasis-open.org/emergency/cap/v1.2/CAP-v1.2.html)